# Notebook 00: Setup Verification

**Time:** 5 minutes  
**Goal:** Verify all packages, system dependencies, and API keys are properly configured

Run each cell below. All checks should show a checkmark. If any fail, follow the fix instructions.

## 1. Python Version

In [1]:
import sys
v = sys.version_info
print(f"Python {v.major}.{v.minor}.{v.micro}")
assert v.major == 3 and v.minor >= 9, "Python 3.9+ required"
print("OK: Python version")

Python 3.11.15
OK: Python version


## 2. Core Packages

In [2]:
core_packages = [
    ("jupyter", "jupyter"),
    ("anthropic", "anthropic>=0.40.0"),
    ("requests", "requests"),
    ("dotenv", "python-dotenv"),
    ("pydantic", "pydantic>=2.0.0"),
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
]

for mod, pip_name in core_packages:
    try:
        __import__(mod)
        print(f"  OK: {mod}")
    except ImportError:
        print(f"  MISSING: {mod} --> pip install {pip_name}")

  OK: jupyter
  MISSING: anthropic --> pip install anthropic>=0.40.0
  OK: requests
  MISSING: dotenv --> pip install python-dotenv
  MISSING: pydantic --> pip install pydantic>=2.0.0
  MISSING: numpy --> pip install numpy
  MISSING: matplotlib --> pip install matplotlib
  MISSING: pandas --> pip install pandas


## 3. Week 3 Packages

In [3]:
week3_packages = [
    # Web scraping
    ("trafilatura", "trafilatura"),
    ("bs4", "beautifulsoup4"),
    # OCR
    ("pytesseract", "pytesseract"),
    ("PIL", "Pillow"),
    # ASR
    ("faster_whisper", "faster-whisper"),
    # TTS
    ("edge_tts", "edge-tts"),
    # Data cleaning
    ("datasketch", "datasketch"),
    ("langdetect", "langdetect"),
    # Utilities
    ("nest_asyncio", "nest-asyncio"),
    ("soundfile", "soundfile"),
]

missing = []
for mod, pip_name in week3_packages:
    try:
        __import__(mod)
        print(f"  OK: {mod}")
    except ImportError:
        print(f"  MISSING: {mod} --> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print(f"\nInstall all missing: pip install {' '.join(missing)}")
else:
    print("\nAll Week 3 packages installed!")

  MISSING: trafilatura --> pip install trafilatura
  OK: bs4
  MISSING: pytesseract --> pip install pytesseract
  MISSING: PIL --> pip install Pillow
  MISSING: faster_whisper --> pip install faster-whisper
  MISSING: edge_tts --> pip install edge-tts
  MISSING: datasketch --> pip install datasketch
  MISSING: langdetect --> pip install langdetect
  OK: nest_asyncio
  MISSING: soundfile --> pip install soundfile

Install all missing: pip install trafilatura pytesseract Pillow faster-whisper edge-tts datasketch langdetect soundfile


## 4. Optional Advanced Packages

These are optional. Install them for bonus notebooks or deeper exploration.

In [4]:
optional_packages = [
    ("crawl4ai", "crawl4ai", "Modern LLM-ready web scraper"),
    ("whisper", "openai-whisper", "OpenAI Whisper (baseline ASR)"),
    ("kokoro", "kokoro", "Kokoro TTS (82M lightweight model)"),
    ("surya", "easyocr", "Surya layout-aware OCR"),
    ("pipecat", "pipecat-ai", "Voice agent framework"),
]

for mod, pip_name, desc in optional_packages:
    try:
        __import__(mod)
        print(f"  OK: {mod} -- {desc}")
    except ImportError:
        print(f"  OPTIONAL: {mod} -- {desc}")
        print(f"            pip install {pip_name}")

  OPTIONAL: crawl4ai -- Modern LLM-ready web scraper
            pip install crawl4ai
  OPTIONAL: whisper -- OpenAI Whisper (baseline ASR)
            pip install openai-whisper
  OPTIONAL: kokoro -- Kokoro TTS (82M lightweight model)
            pip install kokoro
  OPTIONAL: surya -- Surya layout-aware OCR
            pip install surya-ocr
  OPTIONAL: pipecat -- Voice agent framework
            pip install pipecat-ai


## 5. System Dependencies

In [5]:
import subprocess

sys_deps = [
    ("tesseract", "tesseract --version", "brew install tesseract (macOS) / apt install tesseract-ocr (Linux)"),
    ("ffmpeg", "ffmpeg -version", "brew install ffmpeg (macOS) / apt install ffmpeg (Linux)"),
]

for name, cmd, fix in sys_deps:
    try:
        result = subprocess.run(cmd.split(), capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            version = result.stdout.split('\n')[0][:60]
            print(f"  OK: {name} -- {version}")
        else:
            print(f"  MISSING: {name} --> {fix}")
    except (FileNotFoundError, subprocess.TimeoutExpired):
        print(f"  MISSING: {name} --> {fix}")

  OK: tesseract -- tesseract 5.5.2
  OK: ffmpeg -- ffmpeg version 8.1 Copyright (c) 2000-2026 the FFmpeg develo


## 6. Environment Variables

In [6]:
import os
from pathlib import Path
from dotenv import load_dotenv

parent_dir = str(Path(os.getcwd()).parent)
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

env_vars = [
    ("ANTHROPIC_API_KEY", True),
    ("HF_TOKEN", False),
    ("OPENAI_API_KEY", False),
]

for var, required in env_vars:
    value = os.getenv(var)
    label = "REQUIRED" if required else "optional"
    if value:
        masked = value[:8] + "..." + value[-4:]
        print(f"  OK: {var} = {masked}")
    elif required:
        print(f"  MISSING ({label}): {var} -- add to .env file")
    else:
        print(f"  Not set ({label}): {var}")

  OK: ANTHROPIC_API_KEY = sk-ant-a...WgAA
  OK: HF_TOKEN = hf_SopwU...rklv
  OK: OPENAI_API_KEY = sk-proj-...roAA


## 7. Ollama Check

In [7]:
import requests

try:
    resp = requests.get('http://localhost:11434/api/tags', timeout=3)
    if resp.status_code == 200:
        models = [m['name'] for m in resp.json().get('models', [])]
        print(f"  OK: Ollama running with {len(models)} model(s)")
        for m in models:
            flag = " <-- default" if m == "qwen3.5:27b" else ""
            print(f"       - {m}{flag}")
        if 'qwen3.5:27b' not in models:
            print(f"  TIP: Pull default model: ollama pull qwen3.5:27b")
    else:
        print(f"  WARNING: Ollama returned HTTP {resp.status_code}")
except requests.exceptions.RequestException:
    print("  INFO: Ollama not running (needed only for Path B/C)")
    print("        Start with: ollama serve")

  OK: Ollama running with 2 model(s)
       - qwen3.5:27b <-- default
       - qwen3.5:9b


## 8. Directory Structure

In [8]:
import os
from pathlib import Path

parent = Path(os.getcwd()).parent
expected = [
    "notebooks",
    "src",
    "src/__init__.py",
    "src/llm_client.py",
    "src/cost_tracker.py",
    "src/utils.py",
    "src/config.py",
    "src/scraping_utils.py",
    "src/ocr_utils.py",
    "src/audio_utils.py",
    "src/data_pipeline.py",
    "outputs",
]

for item in expected:
    path = parent / item
    if path.exists():
        print(f"  OK: {item}")
    else:
        print(f"  MISSING: {item}")

  OK: notebooks
  OK: src
  OK: src/__init__.py
  OK: src/llm_client.py
  OK: src/cost_tracker.py
  OK: src/utils.py
  OK: src/config.py
  OK: src/scraping_utils.py
  OK: src/ocr_utils.py
  OK: src/audio_utils.py
  OK: src/data_pipeline.py
  OK: outputs


---

## Setup Complete!

**Troubleshooting:**
- Missing Python packages: `pip install -r requirements.txt`
- Missing tesseract: `brew install tesseract` (macOS) or `apt install tesseract-ocr` (Linux)
- Missing ffmpeg: `brew install ffmpeg` (macOS) or `apt install ffmpeg` (Linux)
- Ollama not running: `ollama serve` then `ollama pull qwen3.5:27b`
- Missing .env: `cp .env.example .env` then add your API key

**Next:** Open **Notebook 01: Environment Setup**